In [5]:
from datasets import load_dataset
from transformers import AutoTokenizer

# Gunakan nama repositori lengkap
dataset = load_dataset("Yelp/yelp_review_full")

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained("bert-base-cased")

# Periksa hasil
print(dataset)
print(dataset["train"][0])

[transformers] PyTorch was not found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.
c:\Users\LENOVO\AppData\Local\Programs\Python\Python312\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\LENOVO\.cache\huggingface\hub\datasets--Yelp--yelp_review_full. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-

DatasetDict({
    train: Dataset({
        features: ['label', 'text'],
        num_rows: 650000
    })
    test: Dataset({
        features: ['label', 'text'],
        num_rows: 50000
    })
})
{'label': 4, 'text': "dr. goldberg offers everything i look for in a general practitioner.  he's nice and easy to talk to without being patronizing; he's always on time in seeing his patients; he's affiliated with a top-notch hospital (nyu) which my parents have explained to me is very important in case something happens and you need surgery; and you can get referrals to see specialists without having to see him first.  really, what more do you need?  i'm sitting here trying to think of any complaints i have about him, but i'm really drawing a blank."}


In [6]:
def tokenize_function(examples):
    return tokenizer(examples["text"], padding="max_length", truncation=True)


tokenized_datasets = dataset.map(tokenize_function, batched=True)

small_train_dataset = tokenized_datasets["train"].shuffle(seed=42).select(range(1000))
small_eval_dataset = tokenized_datasets["test"].shuffle(seed=42).select(range(1000))

Map: 100%|██████████| 50000/50000 [00:11<00:00, 4231.76 examples/s]


In [8]:
import tensorflow as tf
from transformers import DefaultDataCollator

# Collator membuat NumPy arrays.
# to_tf_dataset kemudian mengubahnya menjadi TensorFlow tensors.
data_collator = DefaultDataCollator(return_tensors="np")

tf_train_dataset = small_train_dataset.to_tf_dataset(
    columns=["attention_mask", "input_ids", "token_type_ids"],
    label_cols="labels",
    shuffle=True,
    collate_fn=data_collator,
    batch_size=8,
)

tf_validation_dataset = small_eval_dataset.to_tf_dataset(
    columns=["attention_mask", "input_ids", "token_type_ids"],
    label_cols="labels",
    shuffle=False,
    collate_fn=data_collator,
    batch_size=8,
)

# Periksa satu batch
for inputs, labels in tf_train_dataset.take(1):
    print("Bentuk input_ids:", inputs["input_ids"].shape)
    print("Bentuk attention_mask:", inputs["attention_mask"].shape)
    print("Bentuk token_type_ids:", inputs["token_type_ids"].shape)
    print("Bentuk labels:", labels.shape)
    print("Tipe input:", type(inputs["input_ids"]))

Bentuk input_ids: (8, 512)
Bentuk attention_mask: (8, 512)
Bentuk token_type_ids: (8, 512)
Bentuk labels: (8,)
Tipe input: <class 'tensorflow.python.framework.ops.EagerTensor'>


In [ ]:
import tensorflow as tf
from transformers import TFAutoModelForSequenceClassification

model = TFAutoModelForSequenceClassification.from_pretrained(
    "bert-base-cased",
    num_labels=5
)

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=5e-5),
    loss=tf.keras.losses.SparseCategoricalCrossentropy(
        from_logits=True
    ),
    metrics=[tf.keras.metrics.SparseCategoricalAccuracy()]
)

model.fit(
    tf_train_dataset,
    validation_data=tf_validation_dataset,
    epochs=3
)